# QCCD SHAW Demo

This notebook shows the basic workflow for the shuttling code: locate a benchmark circuit, build a small QCCD grid model, and run the SHAW/PGS command-line driver on a small input.

In [ ]:
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "examples":
    repo_root = repo_root.parent

repo_root

In [ ]:
from bqskit.shuttling.qccd.benchmark_paths import benchmark_circuits_dir
from bqskit.shuttling.qccd.benchmark_paths import iter_benchmark_circuit_paths
from bqskit.shuttling.qccd.benchmark_paths import resolve_benchmark_circuit_path

benchmark_dir = benchmark_circuits_dir()
small_benchmarks = [path.name for path in iter_benchmark_circuit_paths("*_8*.qasm")]

print(benchmark_dir)
small_benchmarks[:8]

In [ ]:
from bqskit.ir.circuit import Circuit

circuit_path = resolve_benchmark_circuit_path("QAOA_wsq_8_compiled")
circuit = Circuit.from_file(str(circuit_path))

print(f"Loaded {circuit_path.name}")
print(f"qudits={circuit.num_qudits}, operations={circuit.num_operations}")

In [ ]:
from bqskit.compiler.gateset import GateSet
from bqskit.ir.gates import CXGate
from bqskit.ir.gates.parameterized import U3Gate
from bqskit.shuttling.qccd import QCCDMachineModel
from bqskit.shuttling.qccd import create_grid_physical_machine
from bqskit.shuttling.qccd.run_grid_pgs_shaw import TIMING_DATA
from bqskit.shuttling.qccd.run_grid_pgs_shaw import build_assignment
from bqskit.shuttling.qccd.run_grid_pgs_shaw import congestion_rate

physical_model = create_grid_physical_machine(num_cols=3, num_rows=3, trap_capacity=3)
machine_model = QCCDMachineModel(
    gate_set=GateSet({U3Gate(), CXGate()}),
    physical_graph=physical_model,
    multi_qudit_gate_type="FM",
    timing_data=TIMING_DATA,
)

assignment = build_assignment(machine_model, circuit.num_qudits, seed=1234)
print(f"congestion_rate={congestion_rate(machine_model, circuit.num_qudits)}")
assignment

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "bqskit.shuttling.qccd.run_grid_pgs_shaw",
    "QAOA_wsq_8_compiled",
    "--grid-cols",
    "3",
    "--grid-rows",
    "3",
    "--trap-capacity",
    "3",
    "--num-layout-passes",
    "1",
    "--summary-only",
]

subprocess.run(cmd, cwd=repo_root, check=True)